# Part 5c — Read the failed normal training log

ใช้ WSL/kernel `itcs355-lab3` เดิม รัน code cell เดียว

งานที่ตรวจ: `lab5-20261023-070538-normal`
- ผลที่ส่งมา: ingest/validate Completed; train Failed ก่อนถึง evaluate
- อ่านสถานะและดาวน์โหลด log ของ failed child รวม system log หากไม่มี user log
- ตรวจ registry, endpoint และ compute node counts ณ เวลารัน
- เก็บรายงานเดิมใน inspection-history ก่อนเขียน part5-inspection.json ครั้งใหม่

**อ่านอย่างเดียวบน Azure: ไม่ submit/retrain/setup, เปลี่ยนสิทธิ์, commit หรือ cleanup**
ไม่เปิด compute/endpoint ใหม่; หากมีทรัพยากรเดิมทำงานอยู่ cell นี้ไม่หยุดทรัพยากรนั้น
ส่ง `SEND THIS SUMMARY` กลับมา ยังไม่รัน 05 หรือ 05b ซ้ำ


In [1]:
from pathlib import Path
import platform
import subprocess
import sys

if platform.system() != "Linux" or "microsoft" not in platform.release().lower():
    raise RuntimeError("เลือก kernel itcs355-lab3 ฝั่ง WSL ก่อนครับ")
here = Path.cwd().resolve()
lab = next((base / "lab5" for base in (here, *here.parents)
            if (base / "lab5/scripts/inspect_existing.py").is_file()
            and (base / "lab4/README.md").is_file()), None)
if lab is None:
    raise RuntimeError("เปิด notebook จาก repository เดิมก่อนครับ")
import json
state = json.loads((lab / "reports/managed-execution.json").read_text())
if state.get("jobs", {}).get("normal", {}).get("name") != "lab5-20261023-070538-normal":
    raise RuntimeError("State ไม่ตรงกับ normal job ที่ต้องตรวจ ส่งข้อความนี้กลับมาก่อนครับ")
print("Inspect failed normal job only — no setup, training or submission.", flush=True)
with subprocess.Popen(
    [sys.executable, "-u", "-m", "scripts.inspect_existing"], cwd=lab,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
) as task:
    try:
        for line in task.stdout:
            print(line, end="", flush=True)
        exit_code = task.wait()
    except KeyboardInterrupt:
        task.terminate()
        try:
            task.wait(timeout=10)
        except subprocess.TimeoutExpired:
            task.kill()
        raise
if exit_code:
    raise RuntimeError("อ่านผลเดิมไม่สำเร็จ ส่ง error ด้านบน ไม่ต้องรัน training ใหม่ครับ")
print("ส่ง SEND THIS SUMMARY กลับมา แล้วหยุดตรงนี้ครับ", flush=True)


Inspect failed normal job only — no setup, training or submission.
[15:21:43] Read existing job and child statuses once: lab5-20261023-070538-normal
[15:21:51] Read failed child: train
[15:21:51] Download existing logs only (no models/data): a2c7d68f-f3f2-4415-94f1-7dbdaee69fbe
[15:21:58] Read registry versions, endpoint and current compute nodes; no changes

--- SEND THIS SUMMARY ---
{
  "inspected_at": "2026-10-23T08:21:51.207998+00:00",
  "job": "lab5-20261023-070538-normal",
  "variant": "normal",
  "status": "Failed",
  "parent_error": null,
  "source_commit": "2487815e5bb65d7a216daf62366f254042174d32",
  "children": [
    {
      "name": "aebf681a-10f7-4e29-97b3-db0d80904960",
      "step": "ingest",
      "status": "Completed"
    },
    {
      "name": "2274e4f4-e543-428a-8072-302f33f29031",
      "step": "validate",
      "status": "Completed"
    },
    {
      "name": "a2c7d68f-f3f2-4415-94f1-7dbdaee69fbe",
      "step": "train",
      "status": "Failed",
      "error": null